# 02 — Stage-1 Baseline Estimator: validation & residual analysis

Runs **locally** (inference only, no training). Requires the trained checkpoint
at `checkpoints/estimator.pt` (downloaded from Drive after the Colab run).

Goal: confirm the estimator (a) predicts healthy baselines accurately and (b)
produces residual maps whose fault signatures are localized and physically
sensible — the visual proof behind the aggregate residual stats.


In [ ]:
import sys
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.dataset import InfraredSolarDataset as D
from src.estimator import BaselineEstimator
from src.residual import generate_residual

ckpt = torch.load(ROOT/'checkpoints'/'estimator.pt', map_location='cpu')
model = BaselineEstimator(width=ckpt['width']); model.load_state_dict(ckpt['model_state']); model.eval()
print('loaded estimator | best val MAE =', round(ckpt['val_mae'],4))

## 1. Predicted vs. actual baseline (No-Anomaly val)
The estimator predicts each healthy panel's mean intensity. Points should hug the diagonal.

In [ ]:
val = D(str(ROOT/'data'), str(ROOT/'data/splits/noanomaly_val.csv'))
imgs = torch.stack([val[i][0] for i in range(len(val))])
with torch.no_grad():
    pred = model(imgs).numpy()
actual = imgs.mean(dim=(1,2,3)).numpy()
mae = np.abs(pred-actual).mean()
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(actual, pred, s=6, alpha=0.3)
lims=[min(actual.min(),pred.min()), max(actual.max(),pred.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('actual mean intensity'); ax.set_ylabel('predicted baseline'); ax.set_title(f'No-Anomaly val (MAE={mae:.4f})')
plt.tight_layout(); plt.show()

## 2. Per-class residual montages
For each class, three samples shown as **raw** (top) and **residual** (bottom).
Residuals use a diverging colormap centered at 0 (blue<0, red>0). Look for:
- Diode/Shadowing: ~1/3 module hot (red substring)
- Cell/Hot-Spot: small localized red spot
- Soiling: diffuse, low-magnitude
- No-Anomaly: ~flat (near 0 everywhere)

In [ ]:
ORDER = list(D.CLASS_TO_IDX)
fig, axes = plt.subplots(len(ORDER), 6, figsize=(9, 1.5*len(ORDER)))
rng = np.random.default_rng(0)
full = D(str(ROOT/'data'), [str(ROOT/f'data/splits/{p}.csv') for p in
        ('fault_val','noanomaly_val')])
# index samples by class
by_cls = {}
for i,(_,lbl) in enumerate(full.samples): by_cls.setdefault(lbl, []).append(i)
for r, cls in enumerate(ORDER):
    idxs = by_cls.get(D.CLASS_TO_IDX[cls], [])
    pick = rng.choice(idxs, size=min(3,len(idxs)), replace=False) if idxs else []
    for c, i in enumerate(pick):
        img,_ = full[i]
        res = generate_residual(img.unsqueeze(0), model)[0,0].numpy()
        axes[r,2*c].imshow(img[0].numpy(), cmap='inferno'); axes[r,2*c].axis('off')
        m=np.abs(res).max() or 1
        axes[r,2*c+1].imshow(res, cmap='bwr', vmin=-m, vmax=m); axes[r,2*c+1].axis('off')
    axes[r,0].set_ylabel(cls, fontsize=7, rotation=0, ha='right', va='center');
    axes[r,0].axis('on'); axes[r,0].set_xticks([]); axes[r,0].set_yticks([])
fig.suptitle('raw (inferno) | residual (bwr, 0=white)', y=1.001)
plt.tight_layout(); plt.show()

## 3. Per-class residual magnitude
Mean |residual| per class. If Stage-1 works, fault classes sit above No-Anomaly.

In [ ]:
stats={}
for cls in ORDER:
    idxs = by_cls.get(D.CLASS_TO_IDX[cls], [])
    if not idxs: continue
    batch = torch.stack([full[i][0] for i in idxs])
    res = generate_residual(batch, model)
    stats[cls] = float(res.abs().mean())
fig, ax = plt.subplots(figsize=(11,4))
colors=['#4C72B0' if c=='No-Anomaly' else ('#DD8452' if c in D.ENVIRONMENTAL else '#55A868') for c in stats]
ax.bar(list(stats), list(stats.values()), color=colors)
ax.axhline(stats['No-Anomaly'], color='#4C72B0', ls='--', lw=1, label='No-Anomaly level')
ax.set_ylabel('mean |residual|'); ax.set_title('Residual magnitude by class (blue=healthy, orange=environmental, green=electrical)')
plt.xticks(rotation=45, ha='right'); plt.legend(); plt.tight_layout(); plt.show()
for c,v in sorted(stats.items(), key=lambda x:-x[1]): print(f'{c:16s} {v:.4f}')

## Takeaways
- Estimator predicts healthy baselines to ~0.005 MAE; residual DC level ~0 (irradiance level removed).
- Fault classes show higher mean |residual| than No-Anomaly, and residual maps localize where physics predicts.
- **Next (Phase 3):** add the multiplicative IEC step (divide by baseline) and ablate it; feed residuals to the hierarchical classifier. The classes where subtraction alone barely separates (e.g. Soiling) are exactly where multiplicative normalization should help.
